This notebook fits orthos to all replicates of the shendure-calibrated simulations

Imports

In [3]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

Autoreload for dev

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Create a nice large cluster. We will need the resorces.

In [5]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=5,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=5)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

Load the simulation object

In [6]:
#DATA_ROOT="/gpfs/gibbs/pi/reilly/tabula_data"
DATA_ROOT="/home/mcn26/project_pi_skr2/shared/tabula_data"
simu_obj=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251111")

In [7]:
#temporary : obj created w/ older version of code, necessitating this. 
simu_obj.orthos=[]

Fit orthos to all replicates

In [8]:
client.dashboard_link

'http://10.18.22.93:36625/status'

In [9]:
simu_obj.create_orthos_for_all_replicates(client)

scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


Save

In [10]:
simu_obj.save(path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_with_orthos_20251112")

Shut down the cluster

In [11]:
client.close()
cluster.close()